# Extract Products from CSV

Load the giftcart_cleaned.csv file and extract body_html content into a products array.

In [4]:
# Imports
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv
from typing import List
import os

import pandas as pd
load_dotenv()

from huggingface_hub import login
login(token=os.getenv("HF_ACCESS_TOKEN"))

In [6]:
# Read CSV file
df = pd.read_csv('giftcart_cleaned_products_6000.csv')

# Display basic info
print(f"Total products: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows of description:")
df['description'].head()

Total products: 6082

Columns: ['Id', 'handle', 'title', 'description']

First few rows of description:


0    Make this Valentine’s Day unforgettable for yo...
1    Transform your favorite photo into a fun puzzl...
2    Nutslane Premium Celebration Gift Box | Plain ...
3    Premium Mia Gift Box | Mexican Salsa Cashews |...
4    The Roohani Gift Box is a premium collection o...
Name: description, dtype: str

In [7]:
# Create products array from description column, filtering out NaN values
products = df['description'].dropna().tolist()

print(f"Created products array with {len(products)} items (filtered out empty values)")
print(f"Original row count: {len(df)}")
print(f"\nFirst product preview:")
print(products[0][:500] + "..." if len(products[0]) > 500 else products[0])

Created products array with 6015 items (filtered out empty values)
Original row count: 6082

First product preview:
Make this Valentine’s Day unforgettable for your girlfriend with this charming and thoughtful gift combo. It includes a white ceramic mug, a sweet reminder of how much she means to you every day. and delicious chocolate to add a touch of sweetness to the moment. Beautifully curated with love and care, this romantic combo is a perfect way to express your affection and create lasting memories with the one who holds a special place in your heart. Product Weight: Ceramic Mug - 325 gm Bournville Choc...


In [9]:
# Access individual products
print(f"Total products: {len(products)}")
print(f"\nExample - Product at index 90:")
print(products[90])

Total products: 6015

Example - Product at index 90:
Bring home the cuteness of everyone’s favorite character with this adorable Hello Kitty Soft Toy. Measuring 17 cm x 13 cm, this plush companion is the perfect size for cuddles, décor, or gifting. Made with ultra-soft fabric and fine stitching, it captures Hello Kitty’s signature charm in a huggable form. Whether for kids, collectors, or fans, this soft toy makes a delightful keepsake.


In [10]:
class MyEmbeddings:
    def __init__(self, model):
        self.model = SentenceTransformer(model, trust_remote_code=True)
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self.model.encode(t).tolist() for t in texts]
            
    def embed_query(self, query: str) -> List[float]:
        # Encode single query - returns a single embedding vector
        return self.model.encode(query).tolist()
    
embeddings=MyEmbeddings('google/embeddinggemma-300m')

# Create Document objects with metadata
documents = []
for idx, product in enumerate(products):
    doc = Document(
        page_content=product,
        metadata={"product_id": idx}
    )
    documents.append(doc)

try:
    chromadb = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory="./chroma_db",  # Persist to disk
        collection_name="product_catalog_6000k"
    )
    print(f"Created ChromaDB vector store with {len(documents)} products!")
except Exception as e:
    print(f"Error setting up ChromaDB: {str(e)}")


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: d7e6435c-ea84-43fe-9d5d-e5c57ec53622)')' thrown while requesting HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Created ChromaDB vector store with 6015 products!


# Load Existing DB

In [11]:
class MyEmbeddings:
    def __init__(self, model):
        self.model = SentenceTransformer(model, trust_remote_code=True)
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self.model.encode(t).tolist() for t in texts]
            
    def embed_query(self, query: str) -> List[float]:
        # Encode single query - returns a single embedding vector
        return self.model.encode(query).tolist()
    
embeddings=MyEmbeddings('google/embeddinggemma-300m')
chromadb = Chroma(persist_directory="./chroma_db", embedding_function=embeddings, collection_name="product_catalog_6000k")

In [12]:
# Create retriever
retriever = chromadb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}  # K is the amount of products to return
)

def retriever_tool(query: str) -> str:
    """
    This tool searches and returns relevant products based on the query.
    """
    docs = retriever.invoke(query)

    if not docs:
        return "I found no relevant products."
    
    results = []
    for i, doc in enumerate(docs):
        product_id = doc.metadata.get('product_id', 'Unknown')
        results.append(f"Product {i+1} (ID: {product_id}):\n{doc.page_content}")
    
    return "\n\n".join(results)

In [13]:
retriever_tool(" Initial Stud Earring for Women")

"Product 1 (ID: 1891):\nMcj Jewels stud earrings are charming and perfect for any occasion. They're made with 925 sterling silver and have a cool multi-circle design with a tiny cubic zircon at the top.Plus, they're hypoallergenic, lightweight, and comfortable, so you can wear them all day long at the office or even to gatherings.We made sure to keep all age women and girls in mind when creating these earrings, so they're suitable for everyone. These earrings make a great gift too! They're timeless jewelry that will be loved forever. Number of Items:1Marketed By : Giftcart E-commerce Pvt LtdCustomer Care:+91-9910644899Email: cx@giftcart.comCountry of Origin: India"